In [ ]:
from langgraph.graph import StateGraph,MessagesState,START,END
from langchain_groq import ChatGroq

from langchain_core.messages import HumanMessage,AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import RemoveMessage
from dotenv import load_dotenv
load_dotenv()
import os 
os.environ['LANGCHAIN_PROJECT']="summarise_State"

In [ ]:
model = ChatGroq( model="openai/gpt-oss-20b",
    temperature=0.7)

In [ ]:
class graph_State(MessagesState):
    summary:str

In [ ]:


def chat_node(state:graph_State):
    
    summary = state['summary']
    messages = []
    if summary:
        messages.append({"summary":summary})

    messages.extend(state['messages'])
    res = model.invoke(messages)

    return {"messages":[res.content]}

def summarise(state:graph_State):
    summary = state['summary']

    if summary:
        prompt = f"this is the existing summary-{summary} , extend it using the conversation above.-{state['messages']}"
    else:
        prompt = f"summarise this whole conversation present above.-{state['messages']}"

    res = model.invoke(prompt)

    if len(state['messages'])>6:
        to_delete = state['messages'][:-2]
        return {
            "summary":res.content,
            "messages":[RemoveMessage(id=m.id) for m in to_delete if m is not None]
        }
    return {
        "summary":res.content,
        "messages":[state['messages']]
    }

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
graph = StateGraph(graph_State)

graph.add_node("chat_node",chat_node)
graph.add_node("summarise",summarise)

def should_summarise(state: graph_State):
    if len(state["messages"]) > 6:
        return True
    return False

graph.add_edge(START,"chat_node")
graph.add_conditional_edges(
    "chat_node",
    should_summarise,
    {
        True: "summarise",
        False: END,
    },
)
graph.add_edge("summarise", END)

final_graph = graph.compile(checkpointer=checkpointer)

final_graph

In [ ]:
config = {"configurable":{"thread_id":"thread-1"}}
def show_state():
    snapshot = final_graph.get_state(config)
    values = snapshot.values

    print("\n--- STATE ---")
    print("summary:", values.get("summary", ""))
    print("num_messages:", len(values.get("messages", [])))
    print("messages:")

    for message in values.get("messages", []):
        print(
            "-",
            type(message).__name__,
            ":",
            message.content[:80],
        )

In [ ]:

show_state()
res = final_graph.invoke(
    {
        "messages": [
            HumanMessage(content="What is RDR2?")
        ],
        "summary": ""
    },
    config=config
)

res["messages"]